In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
legal_masks = torch.load("legal_masks.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(legal_masks.shape)

torch.Size([707542, 30, 8, 8])
torch.Size([707542, 4672])


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [6]:
from core import factory

network = factory.build_network("chess")

In [7]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=3e-4, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [8]:
# Train test split
TRAIN_SIZE = int(0.9 * len(states))
assert len(states) == len(policy) == len(value) == len(legal_masks) 

train_states, test_states = states[:TRAIN_SIZE],      states[TRAIN_SIZE:]
train_policy, test_policy = policy[:TRAIN_SIZE],       policy[TRAIN_SIZE:]
train_value,  test_value  = value[:TRAIN_SIZE],        value[TRAIN_SIZE:]
train_masks,  test_masks  = legal_masks[:TRAIN_SIZE],  legal_masks[TRAIN_SIZE:]

assert len(train_states) == len(train_policy) == len(train_value) == len(train_masks)
assert len(test_states)  == len(test_policy)  == len(test_value)  == len(test_masks)

In [12]:
import time
import torch
import numpy as np
from torch import optim
from torch.optim import lr_scheduler
from torch.utils.data import TensorDataset, DataLoader
import itertools

from core.network import PolicyValueNetwork


def evaluate(network, test_states, test_policy, test_value, test_masks,
             policy_loss_fn, value_loss_fn, batch_size=256, eval_samples=4096):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    n = min(len(test_states), eval_samples)

    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]
            batch_mask   = test_masks[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_mask.to(policy_head.device), float("-inf"))

            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()

    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_states: torch.Tensor,
          train_policy: torch.Tensor,
          train_value: torch.Tensor,
          train_masks: torch.Tensor, 
          policy_loss_fn,
          value_loss_fn,
          test_states: torch.Tensor | None = None,
          test_policy: torch.Tensor | None = None,
          test_value: torch.Tensor | None = None,
          test_masks: torch.Tensor | None = None,
          batch_size: int = 256,
          num_epoch: int | None = None,
          duration_hour: float | None = None,
          save_path: str | None = "default.pt"):

    if num_epoch is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_epoch or duration_hour")

    start = time.time()
    step = 0
    best_val_loss = float('inf') # Initialize best validation loss tracking

    # Dataset
    dataset = TensorDataset(train_states, train_policy, train_value.unsqueeze(-1), train_masks)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    total_steps = len(dataloader) * num_epoch if num_epoch is not None else None

    print(len(dataloader))

    warmup_steps = len(dataloader) // 10
    warmup = lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps if total_steps else len(dataloader) * duration_hour * 2)
    scheduler = lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    has_test = test_states is not None and test_policy is not None and test_value is not None
    if has_test:
        test_states = test_states.to(device=DEVICE)
        test_policy = test_policy.to(device=DEVICE)
        test_value  = test_value.unsqueeze(-1).to(device=DEVICE)

    epoch_iter = range(num_epoch) if num_epoch is not None else itertools.count()


    network.save(path=save_path)
    for _ in epoch_iter:
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        for batch_states, batch_policy, batch_value, batch_masks in dataloader:
            batch_states = batch_states.to(DEVICE)
            batch_policy = batch_policy.to(DEVICE)
            batch_value  = batch_value.to(DEVICE)
            batch_masks  = batch_masks.to(DEVICE)

            optimizer.zero_grad()
            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_masks, float("-inf"))

            policy_loss = policy_loss_fn(policy_head, batch_policy)
            value_loss  = value_loss_fn(value_head, batch_value)
            loss = policy_loss + value_loss
            loss.backward()

            optimizer.step()
            scheduler.step()

            if step % 1 == 0:
                elapsed = time.time() - start
                print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={(value_loss.item()):.4f} | lr: {scheduler.get_last_lr()[0]:.8f} | {elapsed:.0f}s")

            step += 1

        if has_test:
            val_policy_loss, val_value_loss = evaluate(
                network, test_states, test_policy, test_value, test_masks,
                policy_loss_fn, value_loss_fn
            )
            
            current_val_loss = val_policy_loss + val_value_loss
            
            print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f} | total_val={current_val_loss:.4f}")

            # Check if this is the best model we've seen so far
            if save_path is not None and current_val_loss < best_val_loss:
                print(f"    [save] Validation loss improved from {best_val_loss:.4f} to {current_val_loss:.4f}. Saving model to '{save_path}'...")
                best_val_loss = current_val_loss
                network.save(path=save_path)


In [ ]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    train_states=train_states,
    train_policy=train_policy,
    train_value=train_value,
    train_masks=train_masks,
    test_states=test_states,
    test_policy=test_policy,
    test_value=test_value,
    test_masks=test_masks,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=512,
)

1244
Network saved at default.pt
[0] loss=4.4981 | policy=3.9086 | value=0.5895 | lr: 0.00000540 | 1s
[1] loss=4.4104 | policy=3.8323 | value=0.5782 | lr: 0.00000779 | 6s
[2] loss=4.4833 | policy=3.9053 | value=0.5781 | lr: 0.00001019 | 11s
[3] loss=4.3011 | policy=3.7770 | value=0.5241 | lr: 0.00001258 | 15s
[4] loss=4.3419 | policy=3.8316 | value=0.5104 | lr: 0.00001498 | 20s
[5] loss=4.4008 | policy=3.8431 | value=0.5577 | lr: 0.00001737 | 25s
[6] loss=4.3677 | policy=3.7992 | value=0.5685 | lr: 0.00001977 | 30s
[7] loss=4.3294 | policy=3.7628 | value=0.5667 | lr: 0.00002216 | 34s
[8] loss=4.3366 | policy=3.7891 | value=0.5475 | lr: 0.00002456 | 39s
[9] loss=4.2534 | policy=3.6847 | value=0.5687 | lr: 0.00002695 | 43s
[10] loss=4.2796 | policy=3.6937 | value=0.5859 | lr: 0.00002935 | 48s
[11] loss=4.1812 | policy=3.6693 | value=0.5119 | lr: 0.00003174 | 53s
[12] loss=4.3016 | policy=3.7159 | value=0.5857 | lr: 0.00003414 | 58s
[13] loss=4.4459 | policy=3.7857 | value=0.6601 | lr: 0.

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:243: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[123] loss=3.5234 | policy=3.0464 | value=0.4770 | lr: 0.00030000 | 711s
[124] loss=3.6581 | policy=3.1065 | value=0.5516 | lr: 0.00030000 | 716s
[125] loss=3.6030 | policy=3.0835 | value=0.5194 | lr: 0.00030000 | 721s
[126] loss=3.6154 | policy=3.0319 | value=0.5835 | lr: 0.00030000 | 725s
[127] loss=3.5325 | policy=2.9779 | value=0.5547 | lr: 0.00030000 | 729s
[128] loss=3.6242 | policy=3.0457 | value=0.5784 | lr: 0.00030000 | 734s
[129] loss=3.6500 | policy=3.0271 | value=0.6229 | lr: 0.00030000 | 739s
[130] loss=3.5690 | policy=3.0350 | value=0.5339 | lr: 0.00029999 | 744s
[131] loss=3.5837 | policy=3.0278 | value=0.5558 | lr: 0.00029999 | 748s
[132] loss=3.6000 | policy=3.0619 | value=0.5382 | lr: 0.00029999 | 753s
[133] loss=3.6295 | policy=3.0985 | value=0.5310 | lr: 0.00029999 | 758s
[134] loss=3.5965 | policy=3.0393 | value=0.5573 | lr: 0.00029999 | 762s
[135] loss=3.5998 | policy=3.0406 | value=0.5591 | lr: 0.00029998 | 767s
[136] loss=3.5319 | policy=2.9990 | value=0.5329 | 

In [ ]:
import numpy as np
avg_legal = test_masks[:4096].sum(dim=1).float().mean().item()
print("uniform-over-legal baseline:", np.log(avg_legal))